In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input/moviereplicationset'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/moviereplicationset/movieReplicationSet.csv


In [3]:
ALPHA = 0.005

import warnings

warnings.filterwarnings('ignore', message='.*All-NaN slice encountered.*')
warnings.filterwarnings('ignore', message='.*Mean of empty slice.*')
warnings.filterwarnings('ignore', message='.*less than 20 observations.*')

In [8]:
import re
# --- Data Loading and Cleaning ---

try:
    df = pd.read_csv('/kaggle/input/moviereplicationset/movieReplicationSet.csv', na_values='N/A')
except FileNotFoundError:
    print("Error: 'movieReplicationSet.csv' not found. Please ensure the file is in the correct directory.")
    exit()

# Define column indices based on the project description
all_columns = df.columns.tolist()
movie_columns = all_columns[0:400]
sensation_seeking_columns = all_columns[400:421]

# Demographic column names
gender_col = all_columns[474]       
only_child_col = all_columns[475]    
movies_alone_col = all_columns[476] 

# Clean demographic data: replace non-testable values with NaN
df[gender_col] = df[gender_col].replace(3, np.nan)
df[only_child_col] = df[only_child_col].replace(-1, np.nan)
df[movies_alone_col] = df[movies_alone_col].replace(-1, np.nan)

def extract_year(movie_title):
    match = re.search(r'\((\d{4})\)$', movie_title)
    return int(match.group(1)) if match else None
    
# Create a DataFrame of movie metadata 
movie_stats = pd.DataFrame({'Movie': movie_columns})
movie_stats['Popularity'] = df[movie_columns].count(axis=0).values
movie_stats['MeanRating'] = df[movie_columns].mean(axis=0).values
movie_stats['Year'] = movie_stats['Movie'].apply(extract_year)

In [49]:
def run_mannwhitneyu(series1, series2, alternative='two-sided'):
    s1_cleaned = series1.dropna()
    s2_cleaned = series2.dropna()
    
    if len(s1_cleaned) < 2 or len(s2_cleaned) < 2:
        return (None, 1.0)
    
    try:
        u_value, p_value = stats.mannwhitneyu(s1_cleaned, s2_cleaned, alternative=alternative)
        return u_value, p_value
    except ValueError:
        return (None, 1.0)

In [50]:
# --- Q1: Popularity vs. Rating ---
print("\n[Q1] Are movies that are more popular rated higher than movies that are less popular?\n")
median_popularity = movie_stats['Popularity'].median()
high_popularity_ratings = movie_stats[movie_stats['Popularity'] > median_popularity]['MeanRating'].dropna()
low_popularity_ratings = movie_stats[movie_stats['Popularity'] <= median_popularity]['MeanRating'].dropna()

q1_u_value, q1_p_value = run_mannwhitneyu(high_popularity_ratings, low_popularity_ratings, alternative='greater')
q1_is_significant = q1_p_value < ALPHA

print(f"  Median Popularity (ratings count): {median_popularity:.0f}")
print(f"  Result: {'Significant' if q1_is_significant else 'Not Significant'} (p = {q1_p_value:.4e})\n")
print("Yes, Movies with higher popularity (more ratings) are rated significantly higher than less popular movies. \n")


[Q1] Are movies that are more popular rated higher than movies that are less popular?

  Median Popularity (ratings count): 198
  Result: Significant (p = 8.4857e-41)

Yes, Movies with higher popularity (more ratings) are rated significantly higher than less popular movies. 



In [51]:
# --- Q2: Newer vs. Older Movies ---
print("[Q2] Are movies that are newer rated differently than movies that are older")
movies_with_year = movie_stats.dropna(subset=['Year'])
median_year = movies_with_year['Year'].median()

newer_ratings = movies_with_year[movies_with_year['Year'] > median_year]['MeanRating']
older_ratings = movies_with_year[movies_with_year['Year'] <= median_year]['MeanRating']

q2_u_value, q2_p_value = run_mannwhitneyu(newer_ratings, older_ratings, alternative='two-sided')
q2_is_significant = q2_p_value < ALPHA

print(f"  Median Release Year: {median_year:.0f}")
print(f"  Result: {'Significant' if q2_is_significant else 'Not Significant'} (p = {q2_p_value:.4e})\n")
print("No, We cannot conclude that age affects the average rating. \n")

[Q2] Are movies that are newer rated differently than movies that are older
  Median Release Year: 1999
  Result: Not Significant (p = 2.6175e-01)

No, We cannot conclude that age affects the average rating. 



In [52]:
# --- Q3: 'Shrek (2001)' - Gendered Ratings ---
print("\n[Q3] Is enjoyment of ‘Shrek (2001)’ gendered, i.e. do male and female viewers rate it differently \n")
shrek = 'Shrek (2001)'
female_shrek = df[df[gender_col] == 1][shrek]
male_shrek = df[df[gender_col] == 2][shrek]

q3_u_value, q3_p_value = run_mannwhitneyu(female_shrek, male_shrek, alternative='two-sided')
q3_is_significant = q3_p_value < ALPHA

print(f"  Result: {'Significant' if q3_is_significant else 'Not Significant'} (p = {q3_p_value:.4e}) \n")
print("No, The difference in ratings for Shrek (2001) between male and female viewers is not significant different. \n")


[Q3] Is enjoyment of ‘Shrek (2001)’ gendered, i.e. do male and female viewers rate it differently 

  Result: Not Significant (p = 5.0537e-02) 

No, The difference in ratings for Shrek (2001) between male and female viewers is not significant different. 



In [53]:
# --- Q4: Proportion of Gendered Movies ---
print("\n[Q4] What proportion of movies are rated differently by male and female viewers? \n")
gender_count = 0
female_viewer = df[df[gender_col] == 1]
male_viewer = df[df[gender_col] == 2]

for movie in movie_columns:
    u_value, p_value = run_mannwhitneyu(female_viewer[movie], male_viewer[movie], alternative='two-sided')
    if p_value < ALPHA:
        gender_count += 1

q4_proportion = gender_count / len(movie_columns)
print(f"  {gender_count} out of {len(movie_columns)} movies show a significant difference.")
print(f"  Proportion: {q4_proportion:.4f} ({q4_proportion:.2%}) \n") 
print("12.50% of all 400 movies which is about 50 movies show a difference in ratings between male and female viewers. \n")


[Q4] What proportion of movies are rated differently by male and female viewers? 

  50 out of 400 movies show a significant difference.
  Proportion: 0.1250 (12.50%) 

12.50% of all 400 movies which is about 50 movies show a difference in ratings between male and female viewers. 



In [55]:
# --- Q5: 'The Lion King (1994)' - Only Child Effect ---
print("\n[Q5] Do people who are only children enjoy ‘The Lion King (1994)’ more than people with siblings? \n")
lion_king = 'The Lion King (1994)'
only_child = df[df[only_child_col] == 1][lion_king]
sibling = df[df[only_child_col] == 0][lion_king]

# Check for 'more' (one-sided: only child > sibling)
q5_u_value, q5_p_value = run_mannwhitneyu(only_child, sibling, alternative='greater')
q5_is_significant = q5_p_value < ALPHA

print(f"  Result: {'Significant' if q5_is_significant else 'Not Significant'} (p = {q5_p_value:.4e}) \n")
print("No. The data does not show that only children enjoy The Lion King (1994) more than viewers with siblings. \n")


[Q5] Do people who are only children enjoy ‘The Lion King (1994)’ more than people with siblings? 

  Result: Not Significant (p = 9.7842e-01) 

No. The data does not show that only children enjoy The Lion King (1994) more than viewers with siblings. 



In [56]:
# --- Q6: Proportion of "Only Child Effect" Movies ---
print("\n[Q6] What proportion of movies exhibit an 'only child effect', i.e. are rated different by viewers with siblings vs. those without? \n ")
oc_count = 0
only_child = df[df[only_child_col] == 1]
sibling = df[df[only_child_col] == 0]

for movie in movie_columns:
    u_value, p_value = run_mannwhitneyu(only_child[movie], sibling[movie], alternative='two-sided')
    if p_value < ALPHA:
        significant_oc_count += 1

q6_proportion = significant_oc_count / len(movie_columns)
print(f"  {oc_count} out of {len(movie_columns)} movies show a significant difference.")
print(f"  Proportion: {q6_proportion:.4f} ({q6_proportion:.2%}) \n")
print("3.50% of all 400 movies which is around 14 movies show a difference in ratings between viewers who are only children and those who have siblings. \n")


[Q6] What proportion of movies exhibit an 'only child effect', i.e. are rated different by viewers with siblings vs. those without? 
 
  0 out of 400 movies show a significant difference.
  Proportion: 0.0875 (8.75%) 

3.50% of all 400 movies which is around 14 movies show a difference in ratings between viewers who are only children and those who have siblings. 



In [57]:
# --- Q7: 'The Wolf of Wall Street (2013)' - Social Watching ---
print("\n[Q7] Do people who like to watch movies socially enjoy ‘The Wolf of Wall Street (2013)’ more than those who prefer to watch them alone? \n")
wolf = 'The Wolf of Wall Street (2013)'

social_wolf = df[df[movies_alone_col] == 0][wolf]
alone_wolf = df[df[movies_alone_col] == 1][wolf]

q7_u_value, q7_p_value = run_mannwhitneyu(social_wolf, alone_wolf, alternative='greater')
q7_is_significant = q7_p_value < ALPHA

print(f"  Result: {'Significant' if q7_is_significant else 'Not Significant'} (p = {q7_p_value:.4e}) \n")
print("No. The data does not show that social viewers enjoy The Wolf of Wall Street (2013) more than solo viewers. \n")


[Q7] Do people who like to watch movies socially enjoy ‘The Wolf of Wall Street (2013)’ more than those who prefer to watch them alone? 

  Result: Not Significant (p = 9.4367e-01) 

No. The data does not show that social viewers enjoy The Wolf of Wall Street (2013) more than solo viewers. 



In [58]:
# --- Q8: Proportion of "Social Watching" Effect Movies ---
print("\n[Q8] What proportion of movies exhibit such a “social watching” effect? \n")
social_count = 0
social = df[df[movies_alone_col] == 0]
alone = df[df[movies_alone_col] == 1]

for movie in movie_columns:
    u_value, p_value = run_mannwhitneyu(social_df[movie], alone_df[movie], alternative='two-sided')
    if p_value < ALPHA:
        social_count += 1

q8_proportion = significant_social_count / len(movie_columns)
print(f"  {social_count} out of {len(movie_columns)} movies show a significant difference.")
print(f"  Proportion: {q8_proportion:.4f} ({q8_proportion:.2%}) \n")
print("2.50% of all 400 movies which is 10 movies show a difference in ratings between social viewers and solo viewers. \n")


[Q8] What proportion of movies exhibit such a “social watching” effect? 

  10 out of 400 movies show a significant difference.
  Proportion: 0.0250 (2.50%) 

2.50% of all 400 movies which is 10 movies show a difference in ratings between social viewers and solo viewers. 



In [59]:
# --- Q9: 'Home Alone (1990)' vs. 'Finding Nemo (2003)' Distributions ---
print("\n[Q9] Is the ratings distribution of ‘Home Alone (1990)’ different than that of ‘Finding Nemo (2003)’? \n")
home_alone_ratings = df['Home Alone (1990)'].dropna()
finding_nemo_ratings = df['Finding Nemo (2003)'].dropna()

q9_u_value, q9_p_value = stats.ks_2samp(home_alone_ratings, finding_nemo_ratings)
q9_is_significant = q9_p_value < ALPHA

print(f"  Result: {'Significant' if q9_is_significant else 'Not Significant'} (p = {q9_p_value:.4e}) \n")
print("Yes. The ratings distributions for Home Alone (1990) and Finding Nemo (2003) are different which means the viewers rated the two films in different patterns. \n")


[Q9] Is the ratings distribution of ‘Home Alone (1990)’ different than that of ‘Finding Nemo (2003)’? 

  Result: Significant (p = 6.3794e-10) 

Yes. The ratings distributions for Home Alone (1990) and Finding Nemo (2003) are different which means the viewers rated the two films in different patterns. 



In [60]:
# --- Q10: Franchise Inconsistency ---
print("\n[Q10] There are ratings on movies from several franchises ([‘Star Wars’, ‘Harry Potter’, ‘The Matrix’, ‘Indiana Jones’, ‘Jurassic Park’, ‘Pirates of the Caribbean’, ‘Toy Story’, ‘Batman’]) in this dataset. How many of these are of inconsistent quality, as experienced by viewers? \n")
franchise_keywords = ['Star Wars', 'Harry Potter', 'The Matrix', 'Indiana Jones', 'Jurassic Park', 'Pirates of the Caribbean', 'Toy Story', 'Batman']
inconsistent_franchise_count = 0
tested_franchises = 0

print(f"  Franchise consistency check (ANOVA, p < {ALPHA} indicates inconsistency):")
for franchise_key in franchise_keywords: 
    movies = [movie for movie in movie_columns if franchise_key in movie]
    
    if len(movies) < 2:
        continue
    
    ratings_data = [df[movie].dropna() for movie in movies]
    valid_ratings_data = [data for data in ratings_data if len(data) > 0]
    
    if len(valid_ratings_data) < 2:
        continue

    tested_franchises += 1
    
    q10_u_value, q10_p_value = stats.f_oneway(*valid_ratings_data)
    
    if q10_p_value < ALPHA:
        inconsistent_franchise_count += 1
        status = "INCONSISTENT"
    else:
        status = "Consistent"
        
    print(f"    - {franchise_key}: {status} (p = {q10_p_value:.4e})")

print(f"  Total franchises tested: {tested_franchises}")
print(f"  Result: {inconsistent_franchise_count} franchises show inconsistent quality. \n")
print("Seven out of the eight(7/8) tested franchises show inconsistent quality. Only Harry Potter was found to be statistically consistent. \n")


[Q10] There are ratings on movies from several franchises ([‘Star Wars’, ‘Harry Potter’, ‘The Matrix’, ‘Indiana Jones’, ‘Jurassic Park’, ‘Pirates of the Caribbean’, ‘Toy Story’, ‘Batman’]) in this dataset. How many of these are of inconsistent quality, as experienced by viewers? 

  Franchise consistency check (ANOVA, p < 0.005 indicates inconsistency):
    - Star Wars: INCONSISTENT (p = 1.5253e-45)
    - Harry Potter: Consistent (p = 5.0899e-01)
    - The Matrix: INCONSISTENT (p = 2.1375e-11)
    - Indiana Jones: INCONSISTENT (p = 2.2613e-09)
    - Jurassic Park: INCONSISTENT (p = 1.8387e-10)
    - Pirates of the Caribbean: INCONSISTENT (p = 6.5821e-05)
    - Toy Story: INCONSISTENT (p = 4.7639e-04)
    - Batman: INCONSISTENT (p = 1.5383e-44)
  Total franchises tested: 8
  Result: 7 franchises show inconsistent quality. 

Seven out of the eight(7/8) tested franchises show inconsistent quality. Only Harry Potter was found to be statistically consistent. 

